# Multi Agents

In [7]:
import os
import math
from typing import Optional, Tuple

from PIL import Image

from dotenv import load_dotenv
from huggingface_hub import notebook_login, whoami

from smolagents import CodeAgent, GoogleSearchTool, InferenceClientModel, VisitWebpageTool, tool

In [4]:
load_dotenv();
notebook_login()

print(whoami()["name"])

stepit


In [5]:
@tool
def calculate_cargo_travel_time(
    origin_coords: Tuple[float, float],
    destination_coords: Tuple[float, float],
    cruising_speed_kmh: Optional[float] = 750.0,  # Average speed for cargo planes
) -> float:
    """
    Calculate the travel time for a cargo plane between two points on Earth using great-circle distance.

    Args:
        origin_coords: Tuple of (latitude, longitude) for the starting point
        destination_coords: Tuple of (latitude, longitude) for the destination
        cruising_speed_kmh: Optional cruising speed in km/h (defaults to 750 km/h for typical cargo planes)

    Returns:
        float: The estimated travel time in hours

    Example:
        >>> # Chicago (41.8781° N, 87.6298° W) to Sydney (33.8688° S, 151.2093° E)
        >>> result = calculate_cargo_travel_time((41.8781, -87.6298), (-33.8688, 151.2093))
    """

    def to_radians(degrees: float) -> float:
        return degrees * (math.pi / 180)

    # Extract coordinates
    lat1, lon1 = map(to_radians, origin_coords)
    lat2, lon2 = map(to_radians, destination_coords)

    # Earth's radius in kilometers
    EARTH_RADIUS_KM = 6371.0

    # Calculate great-circle distance using the haversine formula
    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = (
        math.sin(dlat / 2) ** 2
        + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2) ** 2
    )
    c = 2 * math.asin(math.sqrt(a))
    distance = EARTH_RADIUS_KM * c

    # Add 10% to account for non-direct routes and air traffic controls
    actual_distance = distance * 1.1

    # Calculate flight time
    # Add 1 hour for takeoff and landing procedures
    flight_time = (actual_distance / cruising_speed_kmh) + 1.0

    # Format the results
    return round(flight_time, 2)


print(calculate_cargo_travel_time((41.8781, -87.6298), (-33.8688, 151.2093)))

22.82


In [12]:
model = InferenceClientModel(model_id="moonshotai/Kimi-K3", provider="together")

In [13]:
task = """Find all Batman filming locations in the world, calculate the time to transfer via cargo plane to here (we're in Gotham, 40.7128° N, 74.0060° W), and return them to me as a pandas dataframe.
Also give me some supercar factories with the same cargo plane transfer time."""

In [14]:
agent = CodeAgent(
    model=model,
    tools=[GoogleSearchTool(), VisitWebpageTool(), calculate_cargo_travel_time],
    additional_authorized_imports=["pandas"],
    max_steps=20,
)

In [15]:
result = agent.run(task)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Find all Batman filming locations in the world, calculate the time to transfer via cargo plane to here (we're   │
│ in Gotham, 40.7128° N, 74.0060° W), and return them to me as a pandas dataframe.                                │
│ Also give me some supercar factories with the same cargo plane transfer time.                                   │
│                                                                                                                 │
╰─ InferenceClientModel - moonshotai/Kimi-K3 ─────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  batman_locations = web_search(query="Batman movie filming locations list")                                       
  print(batman_locations)                                                                                          
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
## Search Results
0. [The Batman (2022) - Filming & production](https://www.imdb.com/title/tt1877830/locations/?item=lc2044989)
Source: IMDb

Filming locations ; Necropolis Cemetery, Glasgow, Scotland, UK. (Batman and Selina leaving the cemetery) · 89 ; St.
George's Hall, Liverpool, England, UK. ( ...

1. [The Batman (2022) - Filming & production](https://www.imdb.com/title/tt1877830/locations/?item=lc1870027)
Source: IMDb

Filming locations ; Necropolis Cemetery, Glasgow, Scotland, UK. (Batman and Selina leaving the cemetery) · 89 ; St.
George's Hall, Liverpool, England, UK. ( ...

2. [Batman: The Movie filming locations](https://moviemaps.org/movies/2hr)
Source: MovieMaps

Batman: The Movie was filmed in Los Angeles in the United States of America. Bronson Canyon (Griffith Park) 380 
South San Rafael Avenue

3. [r/batman - There are so many places they could film those 
...](https://www.reddit.com/r/batman/comments/1uifw1x/there_are_so_many_places_they_could_film_those/)
Source: Reddit · r/batman

r/TheBatmanFilm - details, timings, dates and locations of filming across Glasgow for. 108 upvotes · 21 comments 
...

4. [12 real-life Batman movie locations every true Bat-fan 
...](https://www.lifestyleasia.com/ind/travel/batman-movie-locations-you-can-visit/)
Date published: Mar 4, 2022
Source: Lifestyle Asia Hong Kong

Most of the filming of Batman movies is done in the Warner Bros studios and across the US, including New York and 
Pittsburgh. Two Temple Place, ...

5. ['The Batman' Filming Locations in Liverpool](https://www.youtube.com/watch?v=HaneodHhs4w)
Source: YouTube · Movie Worm

Just visiting a few locations from &#39;The Batman&#39; (2022) that were done in Liverpool. 'The Batman' Filming 
Locations in Liverpool. ...

6. [The Batman: Part II (2028) - Filming & production](https://www.imdb.com/title/tt19850008/locations/)
Source: IMDb

Filming & production. The Batman: Part II. Jump to. Filming locations (1), Filming dates (1). Edit. Filming 
locations. London, England, UK. Helpful•5. 1 ...

7. [Batman Filming Locations | 1 Locations on Map](https://www.filmaps.com/films/batman-filming-locations)
Source: Filmaps

Discover all 1 filming locations for Batman () in Salt River. Explore interactive maps and photos of every 
real-world location.

8. [The Batman (2022) - Filming & production](https://www.imdb.com/title/tt1877830/locations/?item=lc1868408)
Source: IMDb

Filming locations ; Necropolis Cemetery, Glasgow, Scotland, UK. (Batman and Selina leaving the cemetery) · 89 ; St.
George's Hall, Liverpool, England, UK. ( ...

Out: None

[Step 1: Duration 10.46 seconds| Input tokens: 2,351 | Output tokens: 210]

...

[Step 10: Duration 8.03 seconds| Input tokens: 61,358 | Output tokens: 5,992]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 11 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer((df_batman_sorted, df_factories_sorted))                                                            
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: (                           movie  \
0   The Dark Knight Rises (2012)   
1   The Dark Knight Rises (2012)   
2           Batman Begins (2005)   
3              The Batman (2022)   
4         The Dark Knight (2008)   
5   The Dark Knight Rises (2012)   
6       Batman: The Movie (1966)   
7           Batman Begins (2005)   
8              The Batman (2022)   
9              The Batman (2022)   
10          Batman Begins (2005)   
11        The Dark Knight (2008)   
12                 Batman (1989)   
13                 Batman (1989)   
14                 Batman (1989)   
15        The Dark Knight (2008)   
16             The Batman (2022)   
17  The Dark Knight Rises (2012)   
18        The Dark Knight (2008)   

                                             location      lat       lon  \
0                        New York City, New York, USA  40.7128  -74.0060   
1                       Pittsburgh, Pennsylvania, USA  40.4406  -79.9959   
2                              Chicago, Illinois, USA  41.8781  -87.6298   
3                              Chicago, Illinois, USA  41.8781  -87.6298   
4                              Chicago, Illinois, USA  41.8781  -87.6298   
5                        Los Angeles, California, USA  34.0522 -118.2437   
6   Bronson Canyon, Griffith Park, Los Angeles, Ca...  34.1222 -118.3036   
7                        Vatnajokull Glacier, Iceland  64.4167  -16.8000   
8          Necropolis Cemetery, Glasgow, Scotland, UK  55.8626   -4.2310   
9           St. George's Hall, Liverpool, England, UK  53.4084   -2.9786   
10      Mentmore Towers, Buckinghamshire, England, UK  51.8720   -0.6850   
11  Cardington Airship Sheds, Bedfordshire, Englan...  52.1100   -0.4180   
12  Pinewood Studios, Iver Heath, Buckinghamshire,...  51.5464   -0.5310   
13  Knebworth House, Knebworth, Hertfordshire, Eng...  51.8750   -0.2140   
14  Hatfield House, Hatfield, Hertfordshire, Engla...  51.7614   -0.2100   
15       Battersea Power Station, London, England, UK  51.4818   -0.1448   
16                                London, England, UK  51.5074   -0.1278   
17                          Jodhpur, Rajasthan, India  26.2389   73.0243   
18              Times Square, Causeway Bay, Hong Kong  22.2800  114.1850   

    cargo_transfer_time_hours  
0                        1.00  
1                        1.74  
2                        2.68  
3                        2.68  
4                        2.68  
5                        6.77  
6                        6.78  
7                        7.53  
8                        8.60  
9                        8.81  
10                       9.10  
11                       9.11  
12                       9.13  
13                       9.14  
14                       9.15  
15                       9.17  
16                       9.17  
17                      18.35  
18                      20.01  ,    manufacturer                          factory      lat      lon  \
0  Aston Martin              Gaydon, England, UK  52.1780  -1.4760   
1       McLaren              Woking, England, UK  51.3190  -0.5580   
2         Lotus     Hethel, Norfolk, England, UK  52.5800   1.2100   
3    Koenigsegg                Angelholm, Sweden  56.2428  12.8622   
4       Bugatti                 Molsheim, France  48.5419   7.4930   
5       Porsche  Stuttgart-Zuffenhausen, Germany  48.8289   9.1550   
6       Ferrari                 Maranello, Italy  44.5290  10.8650   
7        Pagani    San Cesario sul Panaro, Italy  44.5600  11.0400   
8   Lamborghini      Sant'Agata Bolognese, Italy  44.6580  11.2100   
9         Rimac           Sveta Nedelja, Croatia  45.7980  15.8110   

   cargo_transfer_time_hours  
0                       9.01  
1                       9.14  
2                       9.24  
3                      10.06  
4                      10.09  
5                      10.23  
6                      10.72  
7                      10.74  
8                      10.75  
9                    

[Step 11: Duration 2.78 seconds| Input tokens: 75,108 | Output tokens: 6,098]

In [16]:
result

(                           movie  \
 0   The Dark Knight Rises (2012)   
 1   The Dark Knight Rises (2012)   
 2           Batman Begins (2005)   
 3              The Batman (2022)   
 4         The Dark Knight (2008)   
 5   The Dark Knight Rises (2012)   
 6       Batman: The Movie (1966)   
 7           Batman Begins (2005)   
 8              The Batman (2022)   
 9              The Batman (2022)   
 10          Batman Begins (2005)   
 11        The Dark Knight (2008)   
 12                 Batman (1989)   
 13                 Batman (1989)   
 14                 Batman (1989)   
 15        The Dark Knight (2008)   
 16             The Batman (2022)   
 17  The Dark Knight Rises (2012)   
 18        The Dark Knight (2008)   
 
                                              location      lat       lon  \
 0                        New York City, New York, USA  40.7128  -74.0060   
 1                       Pittsburgh, Pennsylvania, USA  40.4406  -79.9959   
 2                         

We can improve over it by adding a dedicated planning step:

In [17]:
agent.planning_interval = 4

detailed_report = agent.run(f"""
You're an expert analyst. You make comprehensive reports after visiting many websites.
Don't hesitate to search for many queries at once in a for loop.
For each data point that you find, visit the source url to confirm numbers.

{task}
""")

print(detailed_report)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ You're an expert analyst. You make comprehensive reports after visiting many websites.                          │
│ Don't hesitate to search for many queries at once in a for loop.                                                │
│ For each data point that you find, visit the source url to confirm numbers.                                     │
│                                                                                                                 │
│ Find all Batman filming locations in the world, calculate the time to transfer via cargo plane to here (we're   │
│ in Gotham, 40.7128° N, 74.0060° W), and return them to me as a pandas dataframe.                                │
│ Also give me some supercar factories with the same cargo plane transfer time.                                   │
│                                                                                                                 │
╰─ InferenceClientModel - moonshotai/Kimi-K3 ─────────────────────────────────────────────────────────────────────╯

────────────────────────────────────────────────── Initial plan ───────────────────────────────────────────────────
Here are the facts I know and the plan of action that I will follow to solve the task:
```
None
```

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Error in code parsing:
Your code snippet is invalid, because the regex pattern <code>(.*?)</code> was not found in it.
Here is your code snippet:
I'll start by searching for Batman filming locations comprehensively.</code>
Make sure to include code with the correct pattern, for instance:
Thoughts: Your thoughts
<code>
# Your python code here
</code>
Make sure to provide correct code blobs.

[Step 1: Duration 3.87 seconds| Input tokens: 2,461 | Output tokens: 200]

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 9 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  result = {                                                                                                       
      "batman_filming_locations": df_batman,                                                                       
      "supercar_factories": df_factories                                                                           
  }                                                                                                                
  final_answer(result)                                                                                             
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: {'batman_filming_locations':                                 Film/Series                             
Location                  City/Region  Country  Latitude  Longitude  Cargo transfer time to Gotham (h)
0              The Dark Knight Rises (2012)                   Wall Street / One Liberty Plaza      New York City, 
New York      USA   40.7075   -74.0113                               1.00
1              The Dark Knight Rises (2012)                   Military Park / City Hall steps           Newark, New
Jersey      USA   40.7357   -74.1724                               1.02
2              The Dark Knight Rises (2012)                    Heinz Field / Acrisure Stadium     Pittsburgh, 
Pennsylvania      USA   40.4468   -80.0158                               1.75
3                  Batman v Superman (2016)                          Michigan Central Station            Detroit, 
Michigan      USA   42.3289   -83.0777                               2.14
4                    The Dark Knight (2008)                   Trump Tower (Wayne Enterprises)            Chicago, 
Illinois      USA   41.8887   -87.6265                               2.68
5                      Batman Begins (2005)                              35 East Wacker Drive            Chicago, 
Illinois      USA   41.8864   -87.6269                               2.68
6                         The Batman (2022)           Thompson Center area / 230 S LaSalle St            Chicago, 
Illinois      USA   41.8830   -87.6316                               2.68
7                    The Dark Knight (2008)        Chicago Loop (LaSalle St / Gotham streets)            Chicago, 
Illinois      USA   41.8781   -87.6298                               2.68
8              The Dark Knight Rises (2012)                        Union Station / LA streets      Los Angeles, 
California      USA   34.0562  -118.2365                               6.77
9                Batman TV series (1966-68)                              Warner Bros. Studios          Burbank, 
California      USA   34.1500  -118.3381                               6.78
10  Batman TV series (1966-68) / 60s Batman                           Bronson Caves (Batcave)   Griffith Park, Los 
Angeles      USA   34.1222  -118.3122                               6.78
11                    Batman Forever (1995)                    Sony Pictures Studios (ex-MGM)      Culver City, 
California      USA   34.0173  -118.4017                               6.79
12                 Batman & Robin / various         Greystone Mansion (Wayne Manor interiors)    Beverly Hills, 
California      USA   34.0925  -118.4000                               6.79
13                     Batman Begins (2005)            Svínafellsjökull Glacier (Vatnajökull)                   
Skaftafell  Iceland   64.0000   -16.8833                               7.51
14                        The Batman (2022)                                Glasgow Necropolis                      
Glasgow       UK   55.8624    -4.2310                               8.60
15             The Dark Knight Rises (2012)                               Glasgow city centre                      
Glasgow       UK   55.8609    -4.2514                               8.60
16                        The Batman (2022)            Royal Liver Building (Bat-signal roof)                    
Liverpool       UK   53.4060    -2.9958                               8.81
17                        The Batman (2022)                                  Anfield Cemetery                    
Liverpool       UK   53.4310    -2.9560                               8.81
18                        The Batman (2022)               St George's Hall (Gotham City Hall)                    
Liverpool       UK   53.4084    -2.9800                               8.81
19             The Dark Knight Rises (2012)                       Wollaton Hall (Wayne Manor)                   
Nottingham       UK   52.9479    -1.2098                               9.00
20

[Step 9: Duration 4.47 seconds| Input tokens: 94,175 | Output tokens: 6,532]

{'batman_filming_locations':                                 Film/Series                                          Location                  City/Region  Country  Latitude  Longitude  Cargo transfer time to Gotham (h)
0              The Dark Knight Rises (2012)                   Wall Street / One Liberty Plaza      New York City, New York      USA   40.7075   -74.0113                               1.00
1              The Dark Knight Rises (2012)                   Military Park / City Hall steps           Newark, New Jersey      USA   40.7357   -74.1724                               1.02
2              The Dark Knight Rises (2012)                    Heinz Field / Acrisure Stadium     Pittsburgh, Pennsylvania      USA   40.4468   -80.0158                               1.75
3                  Batman v Superman (2016)                          Michigan Central Station            Detroit, Michigan      USA   42.3289   -83.0777                               2.14
4                    The Dark K

The problem with this approach is that if we ask an agent to combine results of detailed searches, we can end up filling the context, making the process slow and more expensive in terms of tokens.

What we can do is structure the job with multiple agents and separate the memory usage between different tasks. This way:

- Each agent is more focused on its core task.
- Memory separation allow to reduce the input tokens at each step.

In [28]:
model = InferenceClientModel(model_id="meta-llama/Llama-3.3-70B-Instruct", max_tokens=8096)

web_agent = CodeAgent(
    model=model,
    tools=[
        GoogleSearchTool(provider="serpapi"),
        VisitWebpageTool(),
        calculate_cargo_travel_time,
    ],
    name="web_agent",
    description="Browses the web to find information",
    verbosity_level=0,
    max_steps=10,
)

In [32]:
def check_reasoning_and_plot(final_answer, agent_memory):
    multimodal_model = InferenceClientModel(
        model_id="Qwen/Qwen2.5-VL-72B-Instruct",
        max_tokens=8096,
    )
    filepath = "saved_map.png"
    assert os.path.exists(filepath), "Make sure to save the plot under saved_map.png!"
    image = Image.open(filepath)
    prompt = (...)  # unchanged
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": prompt},
                {"type": "image_url",
                 "image_url": {"url": make_image_url(encode_image_base64(image))}},
            ],
        }
    ]
    output = multimodal_model(messages).content
    print("Feedback: ", output)
    if "FAIL" in output:
        raise Exception(output)
    return True

model = InferenceClientModel(
    "openai/gpt-oss-120b",
    provider="cerebras",   # or "groq"; both are very fast for this model
    max_tokens=8096,
    timeout=300,
)
manager_agent = CodeAgent(
    model=model,
    tools=[calculate_cargo_travel_time],
    managed_agents=[web_agent],
    additional_authorized_imports=[
        "geopandas",
        "plotly",
        "shapely",
        "json",
        "pandas",
        "numpy",
    ],
    planning_interval=5,
    verbosity_level=2,
    final_answer_checks=[check_reasoning_and_plot],
    max_steps=15,
)

We can visualize our multi agents system:

In [33]:
manager_agent.visualize()

CodeAgent | openai/gpt-oss-120b
├── ✅ Authorized imports: ['geopandas', 'plotly', 'shapely', 'json', 'pandas', 'numpy']
├── 🛠️ Tools:
│   ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
│   ┃ Name                        ┃ Description                           ┃ Arguments                             ┃
│   ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│   │ calculate_cargo_travel_time │ Calculate the travel time for a cargo │ origin_coords (`array`): Tuple of     │
│   │                             │ plane between two points on Earth     │ (latitude, longitude) for the         │
│   │                             │ using great-circle distance.          │ starting point                        │
│   │                             │                                       │ destination_coords (`array`): Tuple   │
│   │                             │                                       │ of (latitude, longitude) for the      │
│   │                             │                                       │ destination                           │
│   │                             │                                       │ cruising_speed_kmh (`number`):        │
│   │                             │                                       │ Optional cruising speed in km/h       │
│   │                             │                                       │ (defaults to 750 km/h for typical     │
│   │                             │                                       │ cargo planes)                         │
│   │ final_answer                │ Provides a final answer to the given  │ answer (`any`): The final answer to   │
│   │                             │ problem.                              │ the problem                           │
│   └─────────────────────────────┴───────────────────────────────────────┴───────────────────────────────────────┘
└── 🤖 Managed agents:
    └── web_agent | CodeAgent | meta-llama/Llama-3.3-70B-Instruct
        ├── ✅ Authorized imports: []
        ├── 📝 Description: Browses the web to find information
        └── 🛠️ Tools:
            ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
            ┃ Name                        ┃ Description                       ┃ Arguments                         ┃
            ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
            │ web_search                  │ Performs a google web search for  │ query (`string`): The search      │
            │                             │ your query then returns a string  │ query to perform.                 │
            │                             │ of the top search results.        │ filter_year (`integer`):          │
            │                             │                                   │ Optionally restrict results to a  │
            │                             │                                   │ certain year                      │
            │ visit_webpage               │ Visits a webpage at the given url │ url (`string`): The url of the    │
            │                             │ and reads its content as a        │ webpage to visit.                 │
            │                             │ markdown string. Use this to      │                                   │
            │                             │ browse webpages.                  │                                   │
            │ calculate_cargo_travel_time │ Calculate the travel time for a   │ origin_coords (`array`): Tuple of │
            │                             │ cargo plane between two points on │ (latitude, longitude) for the     │
            │                             │ Earth using great-circle          │ starting point                    │
            │                             │ distance.       

In [34]:
manager_agent.run("""
Find all Batman filming locations in the world, calculate the time to transfer via cargo plane to here (we're in Gotham, 40.7128° N, 74.0060° W).
Also give me some supercar factories with the same cargo plane transfer time. You need at least 6 points in total.
Represent this as spatial map of the world, with the locations represented as scatter points with a color that depends on the travel time, and save it to saved_map.png!

Here's an example of how to plot and return a map:
import plotly.express as px
df = px.data.carshare()
fig = px.scatter_map(df, lat="centroid_lat", lon="centroid_lon", text="name", color="peak_hour", size=100,
     color_continuous_scale=px.colors.sequential.Magma, size_max=15, zoom=1)
fig.show()
fig.write_image("saved_image.png")
final_answer(fig)

Never try to process strings using code: when you have a string to read, just print it and you'll see it.
""")

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Find all Batman filming locations in the world, calculate the time to transfer via cargo plane to here (we're   │
│ in Gotham, 40.7128° N, 74.0060° W).                                                                             │
│ Also give me some supercar factories with the same cargo plane transfer time. You need at least 6 points in     │
│ total.                                                                                                          │
│ Represent this as spatial map of the world, with the locations represented as scatter points with a color that  │
│ depends on the travel time, and save it to saved_map.png!                                                       │
│                                                                                                                 │
│ Here's an example of how to plot and return a map:                                                              │
│ import plotly.express as px                                                                                     │
│ df = px.data.carshare()                                                                                         │
│ fig = px.scatter_map(df, lat="centroid_lat", lon="centroid_lon", text="name", color="peak_hour", size=100,      │
│      color_continuous_scale=px.colors.sequential.Magma, size_max=15, zoom=1)                                    │
│ fig.show()                                                                                                      │
│ fig.write_image("saved_image.png")                                                                              │
│ final_answer(fig)                                                                                               │
│                                                                                                                 │
│ Never try to process strings using code: when you have a string to read, just print it and you'll see it.       │
│                                                                                                                 │
╰─ InferenceClientModel - openai/gpt-oss-120b ────────────────────────────────────────────────────────────────────╯

────────────────────────────────────────────────── Initial plan ───────────────────────────────────────────────────
Here are the facts I know and the plan of action that I will follow to solve the task:
```
**1. Facts survey**

### 1.1. Facts given in the task
- Gotham (reference point) coordinates: **40.7128° N, 74.0060° W** (New York City).
- Required cargo‑plane travel‑time function: `calculate_cargo_travel_time(origin_coords, destination_coords, 
cruising_speed_kmh)`.
- Required final‑answer function: `final_answer`.
- Desired output: a world map saved as **saved_map.png** with scatter points colored by travel time.
- Minimum number of points (Batman filming locations + supercar factories): **6**.

### 1.2. Facts to look up
| Fact | Reason | Where to find |
|------|--------|----------------|
| Complete list of **Batman (film/TV) shooting locations** worldwide (city, venue). | Needed to obtain candidate 
sites for Batman filming. | Wikipedia page “List of Batman film locations”, IMDb shooting locations, or official 
production notes. |
| Geographic **coordinates (latitude, longitude)** for each identified Batman filming location. | Required for 
distance and travel‑time calculations. | Use a geocoding service (e.g., OpenStreetMap Nominatim) or public 
databases like GeoNames. |
| List of **super‑car manufacturers** (e.g., Ferrari, Lamborghini, Bugatti, McLaren, Pagani, Koenigsegg, etc.) and 
their **factory locations**. | Needed to obtain supercar factories for the second set of points. | Company 
websites, Wikipedia “List of automobile manufacturers”, or industry articles. |
| Geographic **coordinates** for each super‑car factory. | Required for distance and travel‑time calculations. | 
Same geocoding approach as above. |
| Typical **cruising speed** of a cargo plane (km/h). | Input to `calculate_cargo_travel_time`. If not provided, 
need a standard value (e.g., 750 km/h). | Aviation reference sites (e.g., Boeing, Airbus specifications) or 
Wikipedia “Cargo aircraft”. |
| Color‑mapping scheme for travel time (e.g., continuous scale from low to high). | Needed to set `color` argument 
in Plotly scatter map. | Plotly built‑in continuous color scales (e.g., `px.colors.sequential.Magma`). |
| Plotly‑compatible **DataFrame structure** (columns: latitude, longitude, name/label, travel_time). | Required for
the scatter map call. | Plotly documentation. |

### 1.3. Facts to derive
| Fact | How to obtain |
|------|---------------|
| **Great‑circle distance** between Gotham (NYC) and each candidate location. | Use the 
`calculate_cargo_travel_time` function (it internally computes distance). |
| **Travel time (hours)** for each location using the cargo‑plane speed. | Call `calculate_cargo_travel_time` with 
origin = Gotham coordinates, destination = location coordinates, cruising_speed_kmh = chosen speed. |
| **Color value** for each point based on travel time (e.g., normalized travel time). | Derive from the travel_time
column (e.g., use raw travel_time as the `color` argument). |
| **Scatter‑map figure** (Plotly) and saved image file `saved_map.png`. | Create a DataFrame with the derived 
columns, plot with `px.scatter_map`, set `color` to travel_time, and write image. |

---

**2. Plan**

1. **Gather Batman filming locations**  
   - Use `web_agent` to retrieve a reliable list of Batman movies/TV series filming locations worldwide.  
   - Extract at least three distinct locations (different cities/continents) and record their names.

2. **Gather super‑car factory locations**  
   - Use `web_agent` to collect a list of major super‑car manufacturers and the geographic location of their 
primary production plants.  
   - Extract at least three distinct factory sites (different cities/continents) and record their names.

3. **Geocode all selected locations**  
   - For each Batman location and each super‑car factory, obtain latitude and longitude via a geocoding service 
(e.g., OpenStreetMap Nominatim) using `web_agent` or an API cal

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
We will start by requesting Batman filming locations.Let's call web_agent.We need to actually write code with      
<code>.<code>                                                                                                      
# Step 1: Get Batman filming locations via web_agent                                                               
batman_task = """Find a list of notable Batman movie or TV series filming locations worldwide.                     
Provide at least three distinct locations (city and country) that have been used for shooting Batman related       
productions.                                                                                                       
Return the answer as a plain English list, each item on a separate line, in the format: "Location – Production     
(Year)". For example:                                                                                              
"Chicago, Illinois, USA – Batman Begins (2005)"."""                                                                
batman_locations_result = web_agent(task=batman_task, additional_args={})                                          
print("Batman locations raw result:")                                                                              
print(batman_locations_result)                                                                                     
</code>                                                                                                            

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  .<code>                                                                                                          
  # Step 1: Get Batman filming locations via web_agent                                                             
  batman_task = """Find a list of notable Batman movie or TV series filming locations worldwide.                   
  Provide at least three distinct locations (city and country) that have been used for shooting Batman related     
  productions.                                                                                                     
  Return the answer as a plain English list, each item on a separate line, in the format: "Location – Production   
  (Year)". For example:                                                                                            
  "Chicago, Illinois, USA – Batman Begins (2005)"."""                                                              
  batman_locations_result = web_agent(task=batman_task, additional_args={})                                        
  print("Batman locations raw result:")                                                                            
  print(batman_locations_result)                                                                                   
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

...

Code execution failed at line 'go = plotly.graph_objs' due to: InterpreterError: Forbidden access to module: 
plotly.graph_objs

[Step 15: Duration 1.30 seconds| Input tokens: 167,620 | Output tokens: 12,283]

Reached max steps.

[Step 16: Duration 3.69 seconds| Input tokens: 193,323 | Output tokens: 14,745]

'**Batman\u202ffilming locations & super‑car factories – cargo‑plane travel time to Gotham**\n\n| # | Name (type) | City – Country | Latitude | Longitude | Cargo‑plane travel time to Gotham* (hours) |\n|---|-------------|----------------|----------|-----------|--------------------------------------------|\n| 1 | **London** (Batman) | London – England,\u202fUK | 51.5074 | –0.1278 | 9.17 |\n| 2 | **Liverpool** (Batman) | Liverpool – England,\u202fUK | 53.4084 | –2.9916 | 8.81 |\n| 3 | **Glasgow** (Batman) | Glasgow – Scotland,\u202fUK | 55.8642 | –4.2518 | 8.60 |\n| 4 | **Chicago** (Batman) | Chicago – Illinois,\u202fUSA | 41.8781 | –87.6298 | 2.68 |\n| 5 | **Knebworth** (Batman) | Knebworth – England,\u202fUK | 51.7925 | –0.2105 | 9.15 |\n| 6 | **Hatfield** (Batman) | Hatfield – England,\u202fUK | 51.7538 | –0.1900 | 9.12 |\n| 7 | **Ferrari** (Super‑car) | Maranello – Italy | 44.5035 | 10.8956 | 8.30 |\n| 8 | **Lamborghini** (Super‑car) | Sant’Agata Bolognese – Italy | 44.5475 | 11.2585

In [35]:
manager_agent.python_executor.state["fig"]

KeyError: 'fig'